# Day 6 | Lab 6.3: Document Intelligence Pipeline — Textract → Comprehend → Bedrock

**Duration:** ~1 hour 15 minutes

**Scenario.** SafeGuard Insurance receives thousands of scanned health and motor insurance claim PDFs every day. We build an **end-to-end AWS document-intelligence pipeline**: Textract extracts text + tables + form fields, Comprehend runs NLP (entities, sentiment, PII detection), we redact PII, then Claude Sonnet 4.5 on Bedrock produces an executive summary + a structured JSON record. Polly (audio) is dropped to keep the lab tight; Comprehend will reappear in Module 9 (Lab 9.4) for PII redaction in observability pipelines.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Generate two synthetic insurance claim PDFs (health + motor) so the lab runs without external assets.
2. Run Textract `detect_document_text` + `analyze_document` (TABLES, FORMS) — read raw text, table cells, and key-value pairs.
3. Run Comprehend on the extracted text — entities, sentiment, key phrases, **PII detection**.
4. Redact detected PII before passing the text to the LLM (the pre-LLM safety step).
5. Use Claude Sonnet 4.5 on Bedrock (`converse` API) to produce both an executive summary and a structured JSON extraction.
6. Wrap all four stages into a single reusable `run_claims_intelligence_pipeline()` function and run it on a second document.

**Tools.** AWS Textract · AWS Comprehend · AWS Bedrock (Claude Sonnet 4.5) · `boto3` · `reportlab` (synthetic PDFs)

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q boto3 pandas tabulate Pillow reportlab python-dotenv


## 2. AWS Credentials

Local-venv pattern. Set `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` in your `.env` or shell.


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Initialise AWS Service Clients

In [ ]:
import os
import json
import time
import re
import boto3
import pandas as pd
from io import BytesIO
from collections import defaultdict

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

textract   = boto3.client('textract',         region_name=REGION)
comprehend = boto3.client('comprehend',       region_name=REGION)
bedrock    = boto3.client('bedrock-runtime',  region_name=REGION)

print(f'Region: {REGION}')
print('✅ Textract / Comprehend / Bedrock clients ready')


## 4. Generate Sample Insurance Claim PDFs

Two realistic claim documents are generated programmatically with `reportlab` so the lab runs without external files. Each PDF has form fields, an expense table, and a narrative section — exactly the layout Textract is designed to handle.


In [4]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

def create_health_claim_pdf(filepath="health_claim.pdf"):
    """Generate a realistic health insurance claim form with tables and narrative."""
    c = canvas.Canvas(filepath, pagesize=letter)
    w, h = letter

    # --- Page Header ---
    c.setFont("Helvetica-Bold", 16)
    c.drawString(1*inch, h - 0.8*inch, "SAFEGUARD INSURANCE LTD.")
    c.setFont("Helvetica", 9)
    c.drawString(1*inch, h - 1.05*inch, "CIN: L66010MH2000PLC129408 | IRDAI Reg No: 115")
    c.setFont("Helvetica-Bold", 12)
    c.drawString(1*inch, h - 1.4*inch, "HEALTH INSURANCE CLAIM FORM")
    c.setFont("Helvetica", 10)
    c.drawString(1*inch, h - 1.65*inch, "Claim Reference: CLM-2025-08834 | Date Filed: 12/02/2025")
    c.line(1*inch, h - 1.75*inch, w - 1*inch, h - 1.75*inch)

    y = h - 2.1*inch

    # --- Section A: Policyholder Info (form key-value pairs) ---
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION A: POLICYHOLDER INFORMATION")
    y -= 0.25*inch
    c.setFont("Helvetica", 9.5)
    form_fields = [
        ("Policy Number", "HI-2025-78432"),
        ("Plan Type", "Star Health Premier - Family Floater"),
        ("Sum Insured", "Rs. 10,00,000"),
        ("Policyholder Name", "Rajesh Kumar Sharma"),
        ("Date of Birth", "15/06/1985"),
        ("Gender", "Male"),
        ("Contact Number", "+91-98765-43210"),
        ("Email", "rajesh.sharma@email.com"),
        ("Aadhaar Number", "9876-5432-1098"),
        ("PAN", "ABCPS1234R"),
        ("Address", "42, Vasant Vihar, Sector 12, Gurugram, Haryana 122001"),
        ("Nominee", "Sunita Sharma (Wife) — Contact: +91-87654-32109"),
    ]
    for key, val in form_fields:
        c.drawString(1.2*inch, y, f"{key}: {val}")
        y -= 0.19*inch

    # --- Section B: Hospitalization Details ---
    y -= 0.15*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION B: HOSPITALIZATION DETAILS")
    y -= 0.25*inch
    c.setFont("Helvetica", 9.5)
    hosp_fields = [
        ("Hospital Name", "Max Super Speciality Hospital, Saket, New Delhi"),
        ("Hospital Registration No", "DEL/REG/HSP/2001/4523"),
        ("Admission Date", "02/02/2025"),
        ("Discharge Date", "08/02/2025"),
        ("Duration of Stay", "6 days"),
        ("Treating Doctor", "Dr. Priya Mehta (Cardiology, Reg: DMC-24567)"),
        ("Diagnosis", "Acute Myocardial Infarction — ICD-10: I21.0"),
        ("Procedure", "Percutaneous Coronary Intervention (PCI) with drug-eluting stent"),
        ("Pre-existing Conditions", "Type 2 Diabetes (diagnosed 2019), Hypertension (diagnosed 2020)"),
    ]
    for key, val in hosp_fields:
        c.drawString(1.2*inch, y, f"{key}: {val}")
        y -= 0.19*inch

    # --- Section C: Expense Table ---
    y -= 0.15*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION C: EXPENSE BREAKDOWN")
    y -= 0.25*inch

    # Table header
    c.setFont("Helvetica-Bold", 9)
    c.drawString(1.2*inch, y, "Item")
    c.drawString(4.5*inch, y, "Days/Units")
    c.drawString(5.5*inch, y, "Rate (Rs.)")
    c.drawString(6.5*inch, y, "Amount (Rs.)")
    y -= 0.05*inch
    c.line(1.2*inch, y, 7.3*inch, y)
    y -= 0.17*inch

    c.setFont("Helvetica", 9)
    expenses = [
        ("Room Charges (General Ward)", "4", "5,000/day", "20,000"),
        ("ICU Charges", "2", "12,000/day", "24,000"),
        ("Surgeon & Anesthesia Fees", "1", "—", "85,000"),
        ("Stent (Drug-Eluting, Medtronic)", "1", "—", "1,20,000"),
        ("Medicines & Consumables", "—", "—", "28,500"),
        ("Diagnostic Tests (ECG, Echo, Blood)", "—", "—", "15,200"),
        ("Nursing Charges", "6", "1,500/day", "9,000"),
        ("Ambulance", "1", "—", "2,000"),
        ("Physiotherapy", "3", "800/session", "2,400"),
    ]
    for item, days, rate, amount in expenses:
        c.drawString(1.2*inch, y, item)
        c.drawString(4.7*inch, y, days)
        c.drawString(5.5*inch, y, rate)
        c.drawString(6.5*inch, y, amount)
        y -= 0.17*inch

    c.line(1.2*inch, y + 0.05*inch, 7.3*inch, y + 0.05*inch)
    c.setFont("Helvetica-Bold", 9.5)
    c.drawString(1.2*inch, y, "TOTAL CLAIMED AMOUNT")
    c.drawString(6.5*inch, y, "3,06,100")
    y -= 0.25*inch
    c.setFont("Helvetica", 9)
    c.drawString(1.2*inch, y, "Amount in words: Rupees Three Lakhs Six Thousand One Hundred Only")

    # --- Section D: Patient Narrative ---
    y -= 0.35*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION D: PATIENT STATEMENT")
    y -= 0.22*inch
    c.setFont("Helvetica", 8.5)
    narrative = [
        "I, Rajesh Kumar Sharma, experienced severe chest pain and shortness of breath on the morning of",
        "February 2, 2025 while at my office in Cyber City, Gurugram. My colleague Amit immediately called",
        "an ambulance. I was rushed to Max Hospital, Saket where Dr. Priya Mehta diagnosed me with an acute",
        "myocardial infarction (heart attack). Emergency angioplasty was performed the same evening with a",
        "drug-eluting stent placement. I was in the ICU for 2 days followed by 4 days in the general ward.",
        "",
        "The medical staff at Max Hospital was excellent and I am grateful for the prompt treatment. I am now",
        "recovering at home with prescribed medications including Aspirin, Clopidogrel, Atorvastatin, and",
        "Metoprolol. My follow-up appointment is scheduled for March 5, 2025 with Dr. Mehta.",
        "",
        "However, the total expenses of Rs. 3,06,100 have been a significant financial burden on my family,",
        "especially since I am the sole breadwinner. I was unable to work for 3 weeks and my monthly income",
        "is approximately Rs. 1,50,000. I request SafeGuard Insurance to process this claim at the earliest.",
        "",
        "I declare that the information provided above is true and correct to the best of my knowledge.",
    ]
    for line in narrative:
        c.drawString(1.2*inch, y, line)
        y -= 0.16*inch

    y -= 0.15*inch
    c.setFont("Helvetica", 9.5)
    c.drawString(1.2*inch, y, "Date: 12/02/2025")
    c.drawString(4*inch, y, "Signature: Rajesh Kumar Sharma")

    c.save()
    print(f"✅ Health claim created: {filepath}")
    return filepath


def create_motor_claim_pdf(filepath="motor_claim.pdf"):
    """Generate a motor insurance claim form."""
    c = canvas.Canvas(filepath, pagesize=letter)
    w, h = letter

    c.setFont("Helvetica-Bold", 16)
    c.drawString(1*inch, h - 0.8*inch, "SAFEGUARD INSURANCE LTD.")
    c.setFont("Helvetica", 9)
    c.drawString(1*inch, h - 1.05*inch, "Motor Insurance Division | 24x7 Claims Helpline: 1800-XXX-YYYY")
    c.setFont("Helvetica-Bold", 12)
    c.drawString(1*inch, h - 1.4*inch, "MOTOR INSURANCE CLAIM FORM")
    c.setFont("Helvetica", 10)
    c.drawString(1*inch, h - 1.65*inch, "Claim Reference: MCL-2025-04221 | Date Filed: 05/02/2025")
    c.line(1*inch, h - 1.75*inch, w - 1*inch, h - 1.75*inch)

    y = h - 2.1*inch

    # Section A
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION A: POLICYHOLDER & VEHICLE DETAILS")
    y -= 0.25*inch
    c.setFont("Helvetica", 9.5)
    fields_a = [
        ("Policy Number", "MC-2024-55678"),
        ("Plan Type", "Comprehensive Motor Insurance"),
        ("Insured Declared Value (IDV)", "Rs. 8,45,000"),
        ("Policyholder", "Ananya Deshmukh"),
        ("Contact", "+91-99001-22334 | ananya.d@outlook.com"),
        ("PAN", "ABCPD5678E"),
        ("Driving License No", "MH-0320200012345"),
        ("Vehicle", "2022 Hyundai Creta SX(O) — Diesel — Polar White"),
        ("Registration No", "MH-02-CK-8834"),
        ("Engine No", "D4FC8HA123456"),
        ("Chassis No", "MALA851CLNM123456"),
    ]
    for k, v in fields_a:
        c.drawString(1.2*inch, y, f"{k}: {v}")
        y -= 0.19*inch

    # Section B
    y -= 0.12*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION B: ACCIDENT DETAILS")
    y -= 0.25*inch
    c.setFont("Helvetica", 9.5)
    fields_b = [
        ("Date & Time of Accident", "01/02/2025, approximately 8:30 PM"),
        ("Location", "Mumbai-Pune Expressway, near Lonavala Tunnel Exit (KM 85)"),
        ("FIR Number", "FIR/LNV/2025/0234 — Lonavala Police Station"),
        ("Weather Conditions", "Heavy rain, poor visibility"),
        ("Road Conditions", "Wet, curved section near tunnel exit"),
        ("Third Party Involved", "Yes — TATA truck (MH-04-AB-1234), driver Mr. Suresh Yadav"),
        ("Injuries", "Minor whiplash to driver; passenger (mother, age 62) — fractured wrist"),
    ]
    for k, v in fields_b:
        c.drawString(1.2*inch, y, f"{k}: {v}")
        y -= 0.19*inch

    # Section C - Damage Table
    y -= 0.12*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION C: DAMAGE ASSESSMENT & REPAIR ESTIMATE")
    y -= 0.25*inch
    c.setFont("Helvetica-Bold", 9)
    c.drawString(1.2*inch, y, "Component")
    c.drawString(4*inch, y, "Damage Type")
    c.drawString(5.5*inch, y, "Action")
    c.drawString(6.3*inch, y, "Cost (Rs.)")
    c.line(1.2*inch, y - 0.05*inch, 7.2*inch, y - 0.05*inch)
    y -= 0.22*inch
    c.setFont("Helvetica", 9)
    damages = [
        ("Front bumper", "Cracked/broken", "Replace", "18,500"),
        ("Left headlamp assembly", "Shattered", "Replace", "12,200"),
        ("Bonnet/Hood", "Dented, paint damage", "Repair + Paint", "15,000"),
        ("Left fender", "Severely dented", "Replace", "11,800"),
        ("Radiator", "Punctured, leaking", "Replace", "22,000"),
        ("AC condenser", "Damaged", "Replace", "8,500"),
        ("Windshield", "Cracked (impact)", "Replace", "14,000"),
        ("Airbag (driver side)", "Deployed", "Replace", "35,000"),
        ("Wheel alignment & suspension", "Misaligned", "Repair", "8,000"),
        ("Labour charges", "—", "—", "25,000"),
        ("Towing charges", "—", "—", "4,500"),
    ]
    for comp, dmg, action, cost in damages:
        c.drawString(1.2*inch, y, comp)
        c.drawString(4*inch, y, dmg)
        c.drawString(5.5*inch, y, action)
        c.drawString(6.3*inch, y, cost)
        y -= 0.17*inch

    c.line(1.2*inch, y + 0.05*inch, 7.2*inch, y + 0.05*inch)
    c.setFont("Helvetica-Bold", 9.5)
    c.drawString(1.2*inch, y, "TOTAL REPAIR ESTIMATE")
    c.drawString(6.3*inch, y, "1,74,500")

    # Section D - Narrative
    y -= 0.35*inch
    c.setFont("Helvetica-Bold", 11)
    c.drawString(1*inch, y, "SECTION D: DRIVER STATEMENT")
    y -= 0.22*inch
    c.setFont("Helvetica", 8.5)
    statement = [
        "I, Ananya Deshmukh, was driving from Mumbai to Pune on February 1, 2025 with my mother.",
        "At approximately 8:30 PM, near the Lonavala tunnel exit, a TATA truck ahead of me suddenly",
        "braked due to a stalled vehicle. Due to heavy rain and poor visibility, I could not stop in time",
        "and collided with the rear of the truck at approximately 50 km/h. The airbag deployed on impact.",
        "My mother, sitting in the front passenger seat, fractured her left wrist on the dashboard.",
        "",
        "I am extremely upset and frustrated with the entire situation. The truck had no reflectors and",
        "its tail lights were not working — this is clearly the truck driver's negligence. We were stranded",
        "on the expressway for over 2 hours before the tow truck arrived. I had to arrange separate",
        "transport to take my injured mother to the hospital. The overall experience was traumatic.",
        "",
        "I have filed FIR/LNV/2025/0234 at Lonavala Police Station. I request immediate processing of",
        "this claim so that repairs can begin. My car is currently at Hyundai authorized service center,",
        "Wakad, Pune (Job Card No: WKD-2025-0891).",
    ]
    for line in statement:
        c.drawString(1.2*inch, y, line)
        y -= 0.16*inch

    y -= 0.15*inch
    c.setFont("Helvetica", 9.5)
    c.drawString(1.2*inch, y, "Date: 05/02/2025")
    c.drawString(4*inch, y, "Signature: Ananya Deshmukh")

    c.save()
    print(f"✅ Motor claim created: {filepath}")
    return filepath


# Generate both documents
health_pdf = create_health_claim_pdf()
motor_pdf = create_motor_claim_pdf()


✅ Health claim created: health_claim.pdf
✅ Motor claim created: motor_claim.pdf


## 5. Pipeline Stage 1 — Document Extraction (Textract)

Two Textract APIs in concert:
- `detect_document_text` returns raw lines + per-line OCR confidence.
- `analyze_document(FeatureTypes=['TABLES','FORMS'])` returns key-value pairs and table cells with relationships.


In [5]:
def textract_extract_text(filepath):
    """Extract raw text lines from a document using detect_document_text."""
    with open(filepath, "rb") as f:
        doc_bytes = f.read()

    start = time.time()
    response = textract.detect_document_text(Document={"Bytes": doc_bytes})
    latency = round(time.time() - start, 2)

    lines = []
    for block in response["Blocks"]:
        if block["BlockType"] == "LINE":
            lines.append({
                "text": block["Text"],
                "confidence": round(block["Confidence"], 1)
            })

    full_text = "\n".join([l["text"] for l in lines])
    avg_conf = round(sum(l["confidence"] for l in lines) / len(lines), 1) if lines else 0

    return {
        "full_text": full_text,
        "lines": lines,
        "total_lines": len(lines),
        "avg_confidence": avg_conf,
        "latency_sec": latency,
        "char_count": len(full_text)
    }


def textract_extract_tables_and_forms(filepath):
    """Extract tables and form key-value pairs using analyze_document."""
    with open(filepath, "rb") as f:
        doc_bytes = f.read()

    start = time.time()
    response = textract.analyze_document(
        Document={"Bytes": doc_bytes},
        FeatureTypes=["TABLES", "FORMS"]
    )
    latency = round(time.time() - start, 2)

    blocks = response["Blocks"]
    block_map = {b["Id"]: b for b in blocks}

    # --- Extract Form Key-Value Pairs ---
    def get_text_from_ids(ids, block_map):
        text = ""
        for rid in ids:
            if rid in block_map:
                child = block_map[rid]
                if child["BlockType"] == "WORD":
                    text += child.get("Text", "") + " "
                elif child["BlockType"] == "SELECTION_ELEMENT":
                    text += "[✓]" if child.get("SelectionStatus") == "SELECTED" else "[_]"
        return text.strip()

    key_value_pairs = []
    for block in blocks:
        if block["BlockType"] == "KEY_VALUE_SET" and "KEY" in block.get("EntityTypes", []):
            key_text = ""
            value_text = ""
            # Get KEY text
            for rel in block.get("Relationships", []):
                if rel["Type"] == "CHILD":
                    key_text = get_text_from_ids(rel["Ids"], block_map)
                if rel["Type"] == "VALUE":
                    for vid in rel["Ids"]:
                        val_block = block_map.get(vid, {})
                        for vrel in val_block.get("Relationships", []):
                            if vrel["Type"] == "CHILD":
                                value_text = get_text_from_ids(vrel["Ids"], block_map)
            if key_text:
                key_value_pairs.append({
                    "key": key_text,
                    "value": value_text,
                    "confidence": round(block.get("Confidence", 0), 1)
                })

    # --- Extract Tables ---
    tables = []
    for block in blocks:
        if block["BlockType"] == "TABLE":
            table_cells = []
            for rel in block.get("Relationships", []):
                if rel["Type"] == "CHILD":
                    for cell_id in rel["Ids"]:
                        cell = block_map.get(cell_id, {})
                        if cell.get("BlockType") == "CELL":
                            cell_text = ""
                            for crel in cell.get("Relationships", []):
                                if crel["Type"] == "CHILD":
                                    cell_text = get_text_from_ids(crel["Ids"], block_map)
                            table_cells.append({
                                "row": cell.get("RowIndex", 0),
                                "col": cell.get("ColumnIndex", 0),
                                "text": cell_text,
                                "confidence": round(cell.get("Confidence", 0), 1)
                            })
            if table_cells:
                tables.append(table_cells)

    return {
        "key_value_pairs": key_value_pairs,
        "tables": tables,
        "total_kvps": len(key_value_pairs),
        "total_tables": len(tables),
        "latency_sec": latency
    }


def run_textract_stage(filepath):
    """Complete Textract stage: raw text + tables/forms analysis."""
    print(f"📄 TEXTRACT — Processing: {filepath}")
    print("-" * 50)

    # Raw text extraction
    text_result = textract_extract_text(filepath)
    print(f"  ✅ Text extraction: {text_result['total_lines']} lines, {text_result['char_count']} chars ({text_result['latency_sec']}s)")
    print(f"     Avg confidence: {text_result['avg_confidence']}%")

    # Table/Form analysis
    struct_result = textract_extract_tables_and_forms(filepath)
    print(f"  ✅ Structure analysis: {struct_result['total_kvps']} form fields, {struct_result['total_tables']} tables ({struct_result['latency_sec']}s)")

    return {
        "text": text_result,
        "structure": struct_result,
        "total_latency_sec": round(text_result["latency_sec"] + struct_result["latency_sec"], 2)
    }


print("Textract functions defined ✅")


Textract functions defined ✅


In [6]:
# --- Run Textract on the health claim ---
health_textract = run_textract_stage("health_claim.pdf")


📄 TEXTRACT — Processing: health_claim.pdf
--------------------------------------------------
  ✅ Text extraction: 80 lines, 2407 chars (3.24s)
     Avg confidence: 96.9%
  ✅ Structure analysis: 33 form fields, 1 tables (4.5s)


In [7]:
# --- Preview: Extracted text ---
print("📝 EXTRACTED TEXT (first 1200 chars):")
print("=" * 60)
print(health_textract["text"]["full_text"][:1200])
print("...")


📝 EXTRACTED TEXT (first 1200 chars):
SAFEGUARD INSURANCE LTD.
CIN: L66010MH2000PLC129408 I IRDAI Reg No: 115
HEALTH INSURANCE CLAIM FORM
Claim Reference: CLM-2025-08834 I Date Filed: 12/02/2025
SECTION A: POLICYHOLDER INFORMATION
Policy Number: HI-2025-78432
Plan Type: Star Health Premier - Family Floater
Sum Insured: Rs. 10,00,000
Policyholder Name: Rajesh Kumar Sharma
Date of Birth: 15/06/1985
Gender: Male
Contact Number: +91-98765-43210
Email: rajesh.sharma@email.com
Aadhaar Number: 9876-5432-1098
PAN: ABCPS1234R
Address: 42, Vasant Vihar, Sector 12, Gurugram, Haryana 122001
Nominee: Sunita Sharma (Wife) - Contact: +91-87654-32109
SECTION B: HOSPITALIZATION DETAILS
Hospital Name: Max Super Speciality Hospital, Saket, New Delhi
Hospital Registration No: DEL/REG/HSP/2001/4523
Admission Date: 02/02/2025
Discharge Date: 08/02/2025
Duration of Stay: 6 days
Treating Doctor: Dr. Priya Mehta (Cardiology, Reg: DMC-24567)
Diagnosis: Acute Myocardial Infarction - ICD-10: I21.0
Procedure: Percu

In [8]:
# --- Preview: Form key-value pairs ---
print("📋 FORM FIELDS EXTRACTED:")
print("=" * 60)
kvp_df = pd.DataFrame(health_textract["structure"]["key_value_pairs"])
if not kvp_df.empty:
    display(kvp_df.head(15))
else:
    print("No form fields detected (expected for some PDF formats)")

# --- Preview: Tables ---
print(f"\n📊 TABLES DETECTED: {health_textract['structure']['total_tables']}")
for i, table in enumerate(health_textract["structure"]["tables"]):
    print(f"\nTable {i+1} ({len(table)} cells):")
    table_df = pd.DataFrame(table)
    if not table_df.empty:
        # Pivot cells into a table grid
        pivot = table_df.pivot(index="row", columns="col", values="text").fillna("")
        display(pivot.head(12))


📋 FORM FIELDS EXTRACTED:


,key,value,confidence
0,Date of Birth:,15/06/1985,95.7
1,Pre-existing Conditions:,"Type 2 Diabetes (diagnosed 2019), Hypertension...",91.8
2,Policyholder Name:,Rajesh Kumar Sharma,95.7
3,PAN:,ABCPS1234R,94.4
4,Email:,rajesh.sharma@email.com,95.2
5,Gender:,Male,94.5
6,Discharge Date:,08/02/2025,95.1
7,Aadhaar Number:,9876-5432-1098,95.6
8,Admission Date:,02/02/2025,95.1
9,Contact:,+91-87654-32109,92.5



📊 TABLES DETECTED: 1

Table 1 (44 cells):


col,1,2,3,4
row,,,,
1,Item,Days/Units,Rate (Rs.),Amount (Rs.)
2,Room Charges (General Ward),4,"5,000/day","20,000"
3,ICU Charges,2,"12,000/day","24,000"
4,Surgeon & Anesthesia Fees,1,-,"85,000"
5,"Stent (Drug-Eluting, Medtronic)",1,-,"1,20,000"
6,Medicines & Consumables,-,-,"28,500"
7,"Diagnostic Tests (ECG, Echo, Blood)",-,|,"15,200"
8,Nursing Charges,6,"1,500/day","9,000"
9,Ambulance,1,-,"2,000"


## 6. Pipeline Stage 2 — NLP Analysis (Comprehend)

Comprehend annotates the extracted text with: entities, sentiment, key phrases, and PII. We will use the PII output for the redaction step in Section 7. (In Module 9 Lab 9.4, the same PII detector is wired into an observability pipeline that scrubs LLM logs before they reach LangSmith.)


In [9]:
def run_comprehend_stage(text):
    """Complete Comprehend analysis: entities, sentiment, key phrases, PII."""
    # Comprehend limit: 5000 bytes UTF-8
    max_bytes = 4900
    analysis_text = text
    truncated = False
    if len(text.encode('utf-8')) > max_bytes:
        analysis_text = text[:4500]
        truncated = True

    print(f"🔍 COMPREHEND — Analyzing {len(analysis_text)} chars" + (" (truncated)" if truncated else ""))
    print("-" * 50)
    start = time.time()

    # 1. Entity Extraction
    ent_resp = comprehend.detect_entities(Text=analysis_text, LanguageCode="en")
    entities = [{"text": e["Text"], "type": e["Type"], "score": round(e["Score"], 3)}
                for e in ent_resp["Entities"]]

    # 2. Sentiment Analysis
    sent_resp = comprehend.detect_sentiment(Text=analysis_text, LanguageCode="en")
    sentiment = {
        "overall": sent_resp["Sentiment"],
        "scores": {k: round(v, 3) for k, v in sent_resp["SentimentScore"].items()}
    }

    # 3. Key Phrase Extraction
    kp_resp = comprehend.detect_key_phrases(Text=analysis_text, LanguageCode="en")
    key_phrases = [{"text": p["Text"], "score": round(p["Score"], 3)}
                   for p in kp_resp["KeyPhrases"]]

    # 4. PII Detection
    pii_resp = comprehend.detect_pii_entities(Text=analysis_text, LanguageCode="en")
    pii_entities = [{
        "type": p["Type"],
        "score": round(p["Score"], 3),
        "text": analysis_text[p["BeginOffset"]:p["EndOffset"]],
        "begin": p["BeginOffset"],
        "end": p["EndOffset"]
    } for p in pii_resp["Entities"]]

    latency = round(time.time() - start, 2)

    result = {
        "entities": entities,
        "sentiment": sentiment,
        "key_phrases": key_phrases,
        "pii_entities": pii_entities,
        "latency_sec": latency,
        "truncated": truncated
    }

    print(f"  ✅ Entities: {len(entities)}")
    print(f"  ✅ Key Phrases: {len(key_phrases)}")
    print(f"  ✅ PII Items: {len(pii_entities)}")
    print(f"  ✅ Sentiment: {sentiment['overall']}")
    print(f"  Completed in {latency}s")
    return result


print("Comprehend functions defined ✅")


Comprehend functions defined ✅


In [10]:
# --- Run Comprehend on the health claim ---
health_comprehend = run_comprehend_stage(health_textract["text"]["full_text"])


🔍 COMPREHEND — Analyzing 2407 chars
--------------------------------------------------
  ✅ Entities: 78
  ✅ Key Phrases: 108
  ✅ PII Items: 25
  ✅ Sentiment: NEUTRAL
  Completed in 1.0s


In [11]:
# --- Detailed NLP results ---
print("👤 ENTITY DISTRIBUTION:")
print("=" * 60)
ent_df = pd.DataFrame(health_comprehend["entities"])
if not ent_df.empty:
    for etype, group in ent_df.groupby("type"):
        examples = group.nlargest(3, "score")["text"].tolist()
        print(f"  {etype:18s}: {len(group)} found → {', '.join(examples[:3])}")

print(f"\n📊 SENTIMENT ANALYSIS:")
print("=" * 60)
print(f"  Overall: {health_comprehend['sentiment']['overall']}")
for label, score in health_comprehend['sentiment']['scores'].items():
    bar = "█" * int(score * 40)
    print(f"  {label:10s}: {score:.3f} {bar}")

print(f"\n🔒 PII DETECTED ({len(health_comprehend['pii_entities'])} items):")
print("=" * 60)
if health_comprehend["pii_entities"]:
    pii_df = pd.DataFrame(health_comprehend["pii_entities"])
    for ptype, group in pii_df.groupby("type"):
        samples = [t[:15] + "..." if len(t) > 15 else t for t in group["text"].head(2)]
        print(f"  {ptype:25s}: {len(group)} found → {', '.join(samples)}")


👤 ENTITY DISTRIBUTION:
  DATE              : 9 found → March 5, 2025, 12/02/2025, 08/02/2025
  LOCATION          : 6 found → 42, Saket, Vasant Vihar, Sector 12, Gurugram, Haryana 122001
  ORGANIZATION      : 5 found → Max Super Speciality Hospital, SAFEGUARD INSURANCE LTD., Max Hospital
  OTHER             : 24 found → 9876-5432-1098, +91-98765-43210, +91-87654-32109
  PERSON            : 10 found → Amit, Rajesh Kumar Sharma, Sunita Sharma
  QUANTITY          : 24 found → 2 days, 6 days, 15,200

📊 SENTIMENT ANALYSIS:
  Overall: NEUTRAL
  Positive  : 0.000 
  Negative  : 0.000 
  Neutral   : 0.999 ███████████████████████████████████████
  Mixed     : 0.000 

🔒 PII DETECTED (25 items):
  ADDRESS                  : 5 found → 42, Vasant Viha..., Saket
  DATE_TIME                : 8 found → 12/02/2025, 15/06/1985
  EMAIL                    : 1 found → rajesh.sharma@e...
  IN_AADHAAR               : 1 found → 9876-5432-1098
  IN_NREGA                 : 1 found → ABCPS1234R
  NAME            

## 7. Pipeline Stage 2b — PII Redaction (Pre-LLM Safety Step)

In [12]:
def redact_pii(text, pii_entities):
    """
    Replace detected PII with type-labeled placeholders.
    Processes in reverse order to preserve character positions.
    """
    # Sort by position (reverse) so replacements don't shift offsets
    sorted_pii = sorted(pii_entities, key=lambda x: x["begin"], reverse=True)

    redacted = text
    redaction_count = 0
    for pii in sorted_pii:
        begin, end = pii["begin"], pii["end"]
        pii_type = pii["type"]
        placeholder = f"[{pii_type}_REDACTED]"
        redacted = redacted[:begin] + placeholder + redacted[end:]
        redaction_count += 1

    return redacted, redaction_count


# Redact PII from the extracted text (only the portion Comprehend analyzed)
analysis_text = health_textract["text"]["full_text"][:4500]
redacted_text, num_redacted = redact_pii(analysis_text, health_comprehend["pii_entities"])

print(f"🔒 Redacted {num_redacted} PII items")
print(f"\n--- Sample of redacted text (first 600 chars) ---")
print(redacted_text[:600])
print("...")


🔒 Redacted 25 PII items

--- Sample of redacted text (first 600 chars) ---
SAFEGUARD INSURANCE LTD.
CIN: L66010MH2000PLC129408 I IRDAI Reg No: 115
HEALTH INSURANCE CLAIM FORM
Claim Reference: CLM-2025-08834 I Date Filed: [DATE_TIME_REDACTED]
SECTION A: POLICYHOLDER INFORMATION
Policy Number: HI-2025-78432
Plan Type: Star Health Premier - Family Floater
Sum Insured: Rs. 10,00,000
Policyholder Name: [NAME_REDACTED]
Date of Birth: [DATE_TIME_REDACTED]
Gender: Male
Contact Number: [PHONE_REDACTED]
Email: [EMAIL_REDACTED]
Aadhaar Number: [IN_AADHAAR_REDACTED]
PAN: [IN_NREGA_REDACTED]
Address: [ADDRESS_REDACTED]
Nominee: [NAME_REDACTED] (Wife) - Contact: [PHONE_REDACTED]
S
...


## 8. Pipeline Stage 3 — LLM Summarisation & Structured Extraction (Claude Sonnet 4.5 on Bedrock)

**Refactor note.** Source notebook used `amazon.nova-lite-v1:0`. Per Module 6 plan we use Claude Sonnet 4.5 on Bedrock — same `converse` API, dramatically better instruction-following and JSON-shape adherence on regulated insurance content.


In [ ]:
BEDROCK_MODEL = 'anthropic.claude-sonnet-4-5-20250929-v1:0'

def bedrock_summarize(text: str, comprehend_analysis: dict, model_id: str = BEDROCK_MODEL) -> dict:
    '''Generate an executive claims summary using Bedrock Converse API + Claude.'''
    entity_summary = ''.join(
        f"  - {e['type']}: {e['text']}\n" for e in comprehend_analysis['entities'][:20]
    )
    top_phrases = ', '.join(
        kp['text'] for kp in sorted(comprehend_analysis['key_phrases'],
                                    key=lambda x: x['score'], reverse=True)[:15]
    )
    prompt = f'''You are a senior insurance claims analyst. Analyse this claim document and produce a comprehensive executive summary.

## DOCUMENT TEXT (PII-redacted):
{text[:3500]}

## NLP CONTEXT:
- Sentiment: {comprehend_analysis['sentiment']['overall']}
- Sentiment Scores: {json.dumps(comprehend_analysis['sentiment']['scores'])}
- PII Items Found: {len(comprehend_analysis['pii_entities'])}
- Key Entities:
{entity_summary}
- Top Key Phrases: {top_phrases}

## REPORT SECTIONS:
### 1. CLAIM OVERVIEW
Claim number, policyholder, hospital/service centre, dates, primary diagnosis or damage type.
### 2. FINANCIAL SUMMARY
Total amount claimed, top 3 expense categories, reasonableness check.
### 3. RISK FLAGS
Pre-existing conditions, unusual expenses, missing documents, inconsistencies. State "None" if none.
### 4. CUSTOMER SENTIMENT ASSESSMENT
How distressed/frustrated is the policyholder; suggested communication tone.
### 5. COMPLIANCE NOTES
PII types detected and any regulatory considerations.
### 6. RECOMMENDED ACTION
APPROVE / REVIEW / ESCALATE — with one-sentence justification.

Be professional, concise, data-driven.'''
    print(f'  🤖 Generating executive summary ({model_id})...')
    t0 = time.time()
    response = bedrock.converse(
        modelId=model_id,
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 1500, 'temperature': 0.3, 'topP': 0.9},
    )
    return {
        'summary':       response['output']['message']['content'][0]['text'],
        'latency_sec':   round(time.time() - t0, 2),
        'input_tokens':  response.get('usage', {}).get('inputTokens', 'N/A'),
        'output_tokens': response.get('usage', {}).get('outputTokens', 'N/A'),
    }

def bedrock_extract_structured(text: str, model_id: str = BEDROCK_MODEL) -> dict:
    '''Extract structured JSON claim data using Bedrock Converse API + Claude.'''
    prompt = f'''Extract structured claim data from this insurance document. Return ONLY valid JSON, no prose, no markdown fences.

DOCUMENT TEXT:
{text[:3000]}

Schema:
{{
  "claim_number": "",
  "claim_type": "HEALTH | MOTOR | PROPERTY",
  "policyholder_name": "",
  "policy_number": "",
  "incident_date": "",
  "filing_date": "",
  "hospital_or_service_center": "",
  "diagnosis_or_damage": "",
  "total_amount_claimed": "",
  "currency": "INR",
  "duration_days": "",
  "treating_doctor_or_assessor": "",
  "pre_existing_conditions": [],
  "top_expenses": [{{"item": "", "amount": ""}}],
  "third_party_involved": false,
  "police_report_filed": false,
  "urgency": "LOW | MEDIUM | HIGH"
}}'''
    print(f'  📦 Extracting structured JSON ({model_id})...')
    t0 = time.time()
    response = bedrock.converse(
        modelId=model_id,
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 1200, 'temperature': 0.0, 'topP': 0.9},
    )
    raw = response['output']['message']['content'][0]['text']
    # Strip accidental markdown fences
    cleaned = re.sub(r'^```(?:json)?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        parsed = {'_error': f'JSON parse failed: {e}', '_raw': raw}
    return {
        'structured':   parsed,
        'latency_sec':  round(time.time() - t0, 2),
        'raw':          raw,
    }

def run_bedrock_stage(text: str, comprehend_analysis: dict, model_id: str = BEDROCK_MODEL) -> dict:
    return {
        'summary':    bedrock_summarize(text, comprehend_analysis, model_id),
        'structured': bedrock_extract_structured(text, model_id),
    }

print('✅ Bedrock + Claude stage functions defined')


In [ ]:
# Run Bedrock on the redacted health-claim text
health_bedrock = run_bedrock_stage(redacted_text, health_comprehend)

print('=' * 70)
print('📋 EXECUTIVE CLAIMS SUMMARY')
print('=' * 70)
print(health_bedrock['summary']['summary'])
print(f"\n⏱  {health_bedrock['summary']['latency_sec']}s · in {health_bedrock['summary']['input_tokens']} / out {health_bedrock['summary']['output_tokens']} tokens")


In [ ]:
print('\n📦 STRUCTURED CLAIM DATA (JSON):')
print('=' * 60)
print(json.dumps(health_bedrock['structured']['structured'], indent=2))
print(f"\n⏱  {health_bedrock['structured']['latency_sec']}s")


## 9. Forward Reference — Comprehend in Module 9 (Lab 9.4)

> **What we built here.** Comprehend's `detect_pii_entities` is used **once per document** as a pre-LLM safety step.
>
> **What we will build in Module 9 Lab 9.4.** The same Comprehend PII detector is wired into the **observability pipeline** > — every LLM trace shipped to LangSmith / LangFuse passes through a Comprehend redactor first, so PII never lands in the > tracing store. Same API, very different deployment shape.


## 10. End-to-End Pipeline — One Function Call

Wrap the four stages (Textract → Comprehend → Redaction → Bedrock) into a single reusable function.


In [ ]:
def run_claims_intelligence_pipeline(filepath: str, bedrock_model: str = BEDROCK_MODEL) -> dict:
    '''End-to-end Claims Intelligence Pipeline (Polly stage dropped per Module 6 plan).'''
    pipeline_start = time.time()
    print('🚀' + '=' * 68)
    print(f'   CLAIMS INTELLIGENCE PIPELINE — {filepath}')
    print('=' * 70)

    print('\n📄 STAGE 1/3: Textract — Document Extraction')
    textract_out = run_textract_stage(filepath)

    print('\n🔍 STAGE 2/3: Comprehend — NLP Analysis')
    comprehend_out = run_comprehend_stage(textract_out['text']['full_text'])

    print('\n🔒 STAGE 2b: PII Redaction')
    analysis_text = textract_out['text']['full_text'][:4500]
    redacted, num_redacted = redact_pii(analysis_text, comprehend_out['pii_entities'])
    print(f'  ✅ Redacted {num_redacted} PII items before sending to LLM')

    print('\n🤖 STAGE 3/3: Bedrock + Claude — Summary + Structured JSON')
    bedrock_out = run_bedrock_stage(redacted, comprehend_out, bedrock_model)

    total_time = round(time.time() - pipeline_start, 2)

    return {
        'source_file':   filepath,
        'textract':      textract_out,
        'comprehend':    comprehend_out,
        'pii_redaction': {'items_redacted': num_redacted},
        'bedrock':       bedrock_out,
        'pipeline_total_sec': total_time,
    }

print('✅ Pipeline composer ready')


## 11. Run the Pipeline on the Motor Insurance Claim

In [ ]:
motor_result = run_claims_intelligence_pipeline('motor_claim.pdf')

print('\n' + '=' * 70)
print('📋 MOTOR CLAIM — EXECUTIVE SUMMARY')
print('=' * 70)
print(motor_result['bedrock']['summary']['summary'])


In [ ]:
print('\n📦 MOTOR CLAIM — STRUCTURED DATA:')
print('=' * 60)
print(json.dumps(motor_result['bedrock']['structured']['structured'], indent=2))
print(f"\n⏱  Pipeline total: {motor_result['pipeline_total_sec']}s")


## 12. Side-by-Side Dashboard — Health vs Motor

In [ ]:
# Run health through the unified pipeline (so we have an apples-to-apples record)
health_result = run_claims_intelligence_pipeline('health_claim.pdf')

rows = []
for r in [health_result, motor_result]:
    rows.append({
        'document':          r['source_file'],
        'textract_lines':    r['textract']['text']['total_lines'],
        'avg_ocr_conf':      r['textract']['text']['avg_confidence'],
        'entities':          len(r['comprehend']['entities']),
        'pii_items':         len(r['comprehend']['pii_entities']),
        'sentiment':         r['comprehend']['sentiment']['overall'],
        'pii_redacted':      r['pii_redaction']['items_redacted'],
        'bedrock_in_tok':    r['bedrock']['summary']['input_tokens'],
        'bedrock_out_tok':   r['bedrock']['summary']['output_tokens'],
        'total_pipeline_s':  r['pipeline_total_sec'],
    })
pd.DataFrame(rows)


---
## 13. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Service orchestration over single-call magic** | Three AWS services chained — each one is the right tool for its stage; the LLM is the *last* step, not the first |
| **Textract for layout, Comprehend for NLP** | Don't ask the LLM to OCR a scanned table; Textract is faster, cheaper, and returns geometry |
| **PII redaction is a *pipeline* stage, not an afterthought** | Comprehend → redact → LLM means PII never reaches the model — and never lands in your trace store |
| **Claude on Bedrock for summary + JSON extraction** | One model, two prompts — natural-language summary for humans + strict JSON for downstream systems |
| **Pipeline as a single function** | `run_claims_intelligence_pipeline(file)` is the contract; stages are private. Easy to test, easy to swap implementations |
| **Forward to Module 9** | Comprehend re-appears in Lab 9.4 as a redactor inside the observability pipeline — same primitive, very different deployment |

**Next Lab:** Module 7 Lab 7.1 — LangGraph Routing: Customer Support Router 🧭


## 14. Stretch Exercise (Optional)

1. Add a 4th stage that drops the structured JSON into a DynamoDB table (or local SQLite for the lab) keyed by `claim_number`.
2. Replace the Bedrock summarisation prompt with a `with_structured_output(ClaimReport)` call (LangChain). What changes? What stays the same?
3. Run all three Bedrock-attached calls (summary, structured, validation) in parallel using `concurrent.futures.ThreadPoolExecutor` — measure latency saving.
4. Add a guardrail (Lab 6.3) to the Bedrock summarisation step so the executive summary cannot leak PII even if redaction misses something.
5. Switch the model to `anthropic.claude-haiku-4-5-...` and rerun the dashboard. How much do tokens / latency / cost change for this workload?
